# 🚀 OmniGen AI Studio - Google Colab 16GB Cloud GPU Server
**Free Tesla T4 / A100 GPU • 0% Laptop Resource Usage • Multi-Model & LoRA Engine**

### Instructions:
1. Go to **Runtime -> Change runtime type -> Select T4 GPU**.
2. Run all cells below.
3. Copy the public **Cloudflare Tunnel URL** (`https://xxxx.trycloudflare.com`).
4. In OmniGen Web App, click the **Settings / Backend** icon in the header, paste the URL, and click **Test Ping**!

In [ ]:
# Cell 1: Install Dependencies
!pip install -q diffusers transformers accelerate safetensors sentencepiece protobuf fastapi uvicorn pydantic pycloudflared nest_asyncio python-multipart peft

In [ ]:
# Cell 2: Setup Model Directory (Google Drive or Colab Storage)
import os

# Set USE_GOOGLE_DRIVE to True if you want to store your models in Google Drive permanently
USE_GOOGLE_DRIVE = False 
MODELS_DIR = "/content/drive/MyDrive/OmniGen_Models" if USE_GOOGLE_DRIVE else "/content/Models"

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

os.makedirs(MODELS_DIR, exist_ok=True)
BASE_MODEL_PATH = os.path.join(MODELS_DIR, "crucibleRINGPonyxl_v28.safetensors")
CURRENT_BASE_MODEL_FILE = "crucibleRINGPonyxl_v28.safetensors"
LIGHTNING_PATH = os.path.join(MODELS_DIR, "sdxl_lightning_4step_lora.safetensors")
NEGATIVE_HANDS_PATH = os.path.join(MODELS_DIR, "NEGATIVE_HANDS.safetensors")

# Download Base Checkpoint if not present
if not os.path.exists(BASE_MODEL_PATH):
    print("📥 Downloading CrucibleRING PonyXL v28 (~6.6GB)...")
    !wget -c "https://huggingface.co/Lies/crucibleRINGPonyxl_v28/resolve/main/crucibleRINGPonyxl_v28.safetensors" -O {BASE_MODEL_PATH}

# Download SDXL Lightning 4-step accelerator
if not os.path.exists(LIGHTNING_PATH):
    print("⚡ Downloading SDXL Lightning 4-Step Accelerator...")
    !wget -c "https://huggingface.co/ByteDance/SDXL-Lightning/resolve/main/sdxl_lightning_4step_lora.safetensors" -O {LIGHTNING_PATH}

print(f"✅ Storage ready at {MODELS_DIR}! You can also drag and drop custom LoRAs into this folder.")

In [ ]:
# Cell 3: Load Model into 16GB GPU VRAM
import torch
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler
from safetensors.torch import load_file

print(f"🚀 Loading SDXL Pipeline on {torch.cuda.get_device_name(0)} (16GB VRAM)...")
global pipe
pipe = StableDiffusionXLPipeline.from_single_file(
    BASE_MODEL_PATH,
    torch_dtype=torch.float16,
    use_safetensors=True
).to("cuda")

pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)

# 1. SDXL Lightning Acceleration
if os.path.exists(LIGHTNING_PATH):
    pipe.load_lora_weights(MODELS_DIR, weight_name=os.path.basename(LIGHTNING_PATH))
    pipe.fuse_lora()
    pipe.unload_lora_weights()
    print("⚡ SDXL Lightning accelerator active!")

# 2. NEGATIVE_HANDS Inversion
if os.path.exists(NEGATIVE_HANDS_PATH):
    try:
        sd = load_file(NEGATIVE_HANDS_PATH)
        if "clip_l" in sd and "clip_g" in sd:
            pipe.load_textual_inversion(sd["clip_l"], token="negative_hands", text_encoder=pipe.text_encoder, tokenizer=pipe.tokenizer)
            pipe.load_textual_inversion(sd["clip_g"], token="negative_hands", text_encoder=pipe.text_encoder_2, tokenizer=pipe.tokenizer_2)
            print("🛡️ NEGATIVE_HANDS Dual-CLIP active!")
    except Exception as e:
        print(f"Note: {e}")

In [ ]:
# Cell 4: Launch FastAPI Server & Cloudflare Tunnel
import io, base64, time, json, threading, nest_asyncio
from PIL import Image
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Optional, Union
import uvicorn
from pycloudflared import try_cloudflare

nest_asyncio.apply()
app = FastAPI(title="OmniGen Colab Server")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class Txt2ImgRequest(BaseModel):
    prompt: str
    negative_prompt: Optional[str] = "score_4, score_5, score_6, negative_hands, bad hands, blurry, low quality"
    steps: Optional[int] = 20
    cfg_scale: Optional[float] = 6.5
    width: Optional[int] = 832
    height: Optional[int] = 1216
    seed: Optional[int] = -1
    base_model: Optional[Union[dict, str]] = "crucibleRINGPonyxl_v28.safetensors"
    loras: Optional[List[Union[dict, str]]] = []
    civitai_api_key: Optional[str] = ""
import requests
import gc

def download_civitai_model(download_url, dest_path, api_key):
    if os.path.exists(dest_path): return True
    print(f"📥 Downloading missing model to {os.path.basename(dest_path)}...")
    headers = {}
    if api_key: headers["Authorization"] = f"Bearer {api_key}"
    try:
        response = requests.get(download_url, headers=headers, stream=True)
        if response.status_code == 200:
            with open(dest_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            return True
        else:
            print(f"❌ Failed to download model: HTTP {response.status_code}")
            return False
    except Exception as e:
        print(f"❌ Download error: {e}")
        return False
@app.get("/")
def health():
    all_files = [f for f in os.listdir(MODELS_DIR) if f.endswith('.safetensors')]
    return {
        "status": "online",
        "gpu": torch.cuda.get_device_name(0),
        "vram_total": f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB",
        "models_count": len(all_files),
        "base_model": "CrucibleRING PonyXL v28 (16GB Cloud GPU)"
    }

@app.get("/api/models")
def list_models():
    all_files = [f for f in os.listdir(MODELS_DIR) if f.endswith('.safetensors')]
    return {
        "count": len(all_files),
        "models": [{"id": f, "name": f.replace('.safetensors', '')} for f in all_files]
    }

@app.post("/sdapi/v1/txt2img")
def txt2img(req: Txt2ImgRequest):
    global CURRENT_BASE_MODEL_FILE, pipe
    t0 = time.time()
    seed = req.seed if (req.seed is not None and req.seed >= 0) else int(torch.randint(0, 2**32, (1,)).item())
    generator = torch.Generator("cuda").manual_seed(seed)
    
    req_base_model_file = CURRENT_BASE_MODEL_FILE
    req_base_model_url = None
    if isinstance(req.base_model, dict):
        req_base_model_file = req.base_model.get("fileName", req_base_model_file)
        req_base_model_url = req.base_model.get("downloadUrl")
    elif isinstance(req.base_model, str) and req.base_model:
        req_base_model_file = req.base_model

    if req_base_model_file != CURRENT_BASE_MODEL_FILE:
        model_path = os.path.join(MODELS_DIR, req_base_model_file)
        if not os.path.exists(model_path):
            if req_base_model_url:
                success = download_civitai_model(req_base_model_url, model_path, req.civitai_api_key)
                if not success:
                    return {"error": f"Failed to download {req_base_model_file}. Check CivitAI API Key."}
            else:
                return {"error": f"Model {req_base_model_file} not found locally and no download URL."}
        
        print(f"🔄 Swapping base model from {CURRENT_BASE_MODEL_FILE} to {req_base_model_file}...")
        del pipe
        gc.collect()
        torch.cuda.empty_cache()
        pipe = StableDiffusionXLPipeline.from_single_file(
            model_path, torch_dtype=torch.float16, use_safetensors=True
        ).to("cuda")
        pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
        CURRENT_BASE_MODEL_FILE = req_base_model_file
    
    loaded_adapters = []
    if req.loras:
        for item in req.loras:
            name = item if isinstance(item, str) else item.get("fileName") or item.get("name")
            weight = 0.85 if isinstance(item, str) else float(item.get("weight", 0.85))
            if not name: continue
            
            lora_file = name if name.endswith(".safetensors") else f"{name}.safetensors"
            lora_path = os.path.join(MODELS_DIR, lora_file)
            lora_url = item.get("downloadUrl") if isinstance(item, dict) else None
            
            if not os.path.exists(lora_path) and lora_url:
                download_civitai_model(lora_url, lora_path, req.civitai_api_key)
                
            if os.path.exists(lora_path) and lora_file != CURRENT_BASE_MODEL_FILE and lora_file != os.path.basename(LIGHTNING_PATH):
                try:
                    adapter_id = f"lora_{len(loaded_adapters)}"
                    pipe.load_lora_weights(MODELS_DIR, weight_name=lora_file, adapter_name=adapter_id)
                    pipe.set_adapters([adapter_id], adapter_weights=[weight])
                    loaded_adapters.append(adapter_id)
                except Exception as e:
                    print(f"LoRA load note: {e}")

    prompt_str = req.prompt
    if "score_" not in prompt_str:
        prompt_str = f"score_9, score_8_up, score_7_up, source_anime, {prompt_str}"
        
    with torch.inference_mode():
        image = pipe(
            prompt=prompt_str,
            negative_prompt=req.negative_prompt,
            num_inference_steps=req.steps,
            guidance_scale=req.cfg_scale,
            width=req.width,
            height=req.height,
            generator=generator
        ).images[0]

    if loaded_adapters:
        try:
            pipe.delete_adapters(loaded_adapters)
        except Exception:
            pass

    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return {
        "images": [base64.b64encode(buf.getvalue()).decode("utf-8")],
        "source": f"Google Colab Cloud GPU ({torch.cuda.get_device_name(0)})"
    }
threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"), daemon=True).start()
time.sleep(2)

print("\n" + "="*70)
print("🌐 STARTING PUBLIC CLOUDFLARE TUNNEL...")
tunnel = try_cloudflare(port=8000)
print("="*70)
print("🎉 COPY THIS URL AND PASTE IT INTO YOUR OMNIGEN WEB APP:")
print(f"👉  {tunnel.tunnel}  👈")
print("="*70 + "\n")